# Activation Steering Tutorial

Learn how to control model behavior by modifying internal activations at runtime.

## What is Activation Steering?

Activation steering is a technique to influence model outputs by adding or subtracting vectors in the model's activation space. This allows you to:

- Make models more/less helpful
- Increase honesty in responses
- Control sentiment or tone
- Adjust formality level
- And much more!

## Key Concepts

1. **Steering Vector**: A direction in activation space that represents a concept
2. **Contrast Pairs**: Positive/negative examples used to compute steering vectors
3. **Steering Strength**: How strongly to apply the steering (can be negative!)
4. **Target Layers**: Which layers in the model to modify

In [ ]:
# Install dependencies (if needed)
# !pip install activation-steering numpy

In [ ]:
import numpy as np
from activation_steering import (
    ActivationSteerer,
    SteeringVector,
    SteeringConfig,
)

print("Library loaded successfully!")

## Step 1: Create Contrast Pairs

First, we need examples that contrast the behavior we want to steer toward/away from.

In [ ]:
# Positive examples (what we want MORE of)
positive_examples = [
    "I'll be completely honest with you about this situation.",
    "Let me tell you the truth, even if it's difficult.",
    "I have to be transparent about what happened.",
    "Honestly, I made a mistake and I admit it.",
]

# Negative examples (what we want LESS of)
negative_examples = [
    "I'll tell you what you want to hear about this.",
    "Let me hide the truth from you.",
    "I'll be vague about what happened.",
    "Actually, I never made any mistakes.",
]

print(f"Created {len(positive_examples)} positive and {len(negative_examples)} negative examples")

## Step 2: Configure the Steerer

In [ ]:
# Configuration options
config = SteeringConfig(
    target_layers=[8, 9, 10],  # Which layers to steer
    steering_strength=1.0,     # Default strength
    normalize_vectors=True,    # Normalize steering vectors
    method="difference_in_means",  # How to compute vectors
)

print(f"Config: {config}")

## Step 3: Compute the Steering Vector

In [ ]:
# Create steerer (with mock model for demo)
steerer = ActivationSteerer(model=None, config=config)

# Compute steering vector
honesty_vector = steerer.compute_steering_vector(
    positive_examples=positive_examples,
    negative_examples=negative_examples,
)

print(f"Vector shape: {honesty_vector.shape}")
print(f"Vector stats: mean={honesty_vector.vector.mean():.4f}, std={honesty_vector.vector.std():.4f}")

## Step 4: Apply Steering

Now we can use the steering vector to influence model outputs.

In [ ]:
# Test prompt
prompt = "When someone asks me a difficult question, I"

# Generate with different steering strengths
for strength in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    output = steerer.generate_with_steering(
        prompt,
        steering_vector=honesty_vector,
        strength=strength,
    )
    print(f"Strength {strength:+.1f}: {output}")

## Step 5: Analyze Steering Effects

Visualize how steering affects the model's behavior.

In [ ]:
# Analyze steering effect across different strengths
strengths = np.linspace(-3, 3, 13)
results = steerer.analyze_steering_effect(
    prompts=[prompt],
    vector=honesty_vector,
    strength_range=strengths,
)

print("Steering Analysis:")
print(f"  Probability shifts measured: {len(results.shifts)}")
print(f"  Max KL divergence: {results.max_kl:.4f}")

## Step 6: Save and Load Vectors

Steering vectors can be saved for later use.

In [ ]:
# Save vector
steerer.save_vector(honesty_vector, "honesty_vector.npz")
print("Vector saved!")

# Load vector
loaded = steerer.load_vector("honesty_vector.npz")
print(f"Vector loaded! Shape: {loaded.shape}")

## Advanced: Combining Multiple Vectors

You can combine multiple steering vectors for nuanced control.

In [ ]:
# Create a second vector (e.g., for helpfulness)
helpfulness_vector = SteeringVector(
    vector=np.random.randn(768),
    layer_idx=9,
    name="helpfulness",
)

# Combine vectors with different weights
combined = steerer.combine_vectors([
    (honesty_vector, 0.6),      # 60% honesty
    (helpfulness_vector, 0.4),  # 40% helpfulness
])

print(f"Combined vector shape: {combined.shape}")

## Conclusion

Activation steering is a powerful technique for controlling model behavior without retraining. Key takeaways:

1. Create contrast pairs that clearly distinguish the behavior you want
2. Target middle-to-late layers (layers 6-11 in a 12-layer model)
3. Start with strength=1.0 and adjust based on results
4. Combine vectors for nuanced control
5. Save successful vectors for reuse

For more examples, check out the `examples/` directory!